# 01 · Calidad de datos
Todo el conteo se hace **en el servidor** (`python -m src.extract`); aquí solo se leen los resúmenes.
Decisiones que salen de este notebook:
1. `visit_occurrence` no sirve para medir tiempos (inicio = fin) → los tiempos salen de `visit_detail`.
2. Se reconstruyen **episodios de urgencias** (etapas de la misma persona separadas < 6 h).
3. El análisis se corta el **30-abr-2026**: desde mayo la carga está incompleta.

In [ ]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from src import datos, flow
from src.config import FIGURES, DATA_INTERIM, DATA_PROCESSED, ETAPAS
pd.set_option("display.max_columns", 50); pd.set_option("display.float_format", "{:,.2f}".format)
sns.set_theme(style="whitegrid", context="notebook")
def guardar(nombre):
    plt.tight_layout(); plt.savefig(FIGURES / f"{nombre}.png", dpi=150, bbox_inches="tight"); plt.show()


In [ ]:
cal = pd.read_parquet(DATA_INTERIM / "calidad_intervalos.parquet")
cal["pct_duracion_cero"] = cal.duracion_cero / cal.n * 100
cal

In [ ]:
men = pd.read_parquet(DATA_INTERIM / "mensual_tablas.parquet").dropna()
men["mes"] = pd.to_datetime(men["mes"])
men = men[(men.mes >= "2022-12-01") & (men.mes <= "2026-09-01")]
piv = men.pivot_table(index="mes", columns="tabla", values="n", aggfunc="sum")
piv = piv.rename(columns=lambda c: ETAPAS.get(int(c.split("_")[-1]), c) if c.startswith("visit_detail_") else c)
fig, ax = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
piv[[c for c in piv if c in ETAPAS.values()]].plot(ax=ax[0]); ax[0].set_title("Etapas (visit_detail) por mes")
piv[[c for c in ["condition_occurrence", "measurement"] if c in piv]].plot(ax=ax[1]); ax[1].set_title("Tablas clínicas por mes (procedure_date viene vacío)")
for a in ax: a.axvline(pd.Timestamp("2026-05-01"), color="red", ls="--", lw=1)
guardar("01_volumen_mensual")

In [ ]:
et = datos.etapas()
ep = datos.episodios(et)
resumen = pd.Series({
    "etapas únicas (urgencias + hospitalización)": len(et),
    "episodios": len(ep), "pacientes": ep.person_id.nunique(),
    "etapas con fin nulo": int(et.fin_nulo.sum()),
    "etapas con fin < inicio (corregidas)": int(et.fin_antes_inicio.sum()),
    "episodios hospitalizados": int(ep.hospitalizado.sum()),
    "episodios con estancia atípica (excluidos de modelos)": int(ep.los_atipica.sum()),
    "episodios sin diagnóstico codificado": int(ep.dx_principal.isna().sum()),
    "procedure_date vacío en toda la tabla": "sí → procedimientos asociados por visita",
    "periodo": f"{ep.inicio.min():%Y-%m-%d} → {ep.inicio.max():%Y-%m-%d}",
})
resumen.to_frame("valor")

### Duración por etapa: distribución
Triage y atención de urgencias deberían durar minutos/horas; hospitalización, días. Se revisan colas extremas.

In [ ]:
print(et.groupby("etapa")["duracion_h"].describe(percentiles=[.5,.9,.99]).round(2))
print("\nEpisodios con estancia > 30 días:", int((ep.los_h > 720).sum()))
print("Episodios con estancia = 0:", int((ep.los_h == 0).sum()), f"({(ep.los_h == 0).mean():.1%})")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(ep.loc[ep.tipo_detalle == "Urgencias con observación", "los_h"].clip(upper=72), bins=96, ax=ax[0]); ax[0].set_title("Urgencias con observación (≥15 min), horas (cortado en 72)")
sns.histplot(ep.loc[ep.hospitalizado, "los_h"].clip(upper=24*30)/24, bins=60, ax=ax[1]); ax[1].set_title("Estancia de episodios hospitalizados, días (cortado en 30)")
guardar("01_distribucion_estancia")

### Completitud de variables clínicas por episodio

In [ ]:
comp = pd.Series({c: (ep[c] > 0).mean() * 100 for c in ["n_labs", "n_dx", "n_proc", "n_consultas_amb_30d"]})
comp.round(1).to_frame("% episodios con al menos 1")